# **0. LIBRERÍAS Y CONFIGURACIÓN**

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

# Preprocesamiento y Modelado
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline as SklearnPipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, PowerTransformer
from sklearn.preprocessing import TargetEncoder
# modelos

from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor

# Métricas
import optuna
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score



c:\Users\Zenbook\Documents\Aprendizaje_nube\Proyecto2_ML\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# **1. Carga de datos procesados y exploración inicial**

In [2]:
df = pd.read_parquet("../data/processed/dataset.parquet")



# **Feature engineering**

In [3]:
# Filtro de precios entre 500k y 50M, filtro que se había creado en el EDA toca volver a crearlo aquí para que el modelo no se vea afectado por outliers
df_filtrado = df[(df["Precio"] >= 500_000) & (df["Precio"] <= 50_000_000) & (df["Área Construida (m2)"] <= 2500) & (df["Área Privada (m2)"] <= 2500)].copy()


In [4]:
# Definimos nuevamente piso_cat que quedó en el EDA

def piso_cat(x):
    if pd.isna(x):
        return "missing"
    elif x == 1:
        return "1"
    elif x == 2:
        return "2"
    else:
        return "3+"

df_filtrado["Piso_cat"] = df_filtrado["Piso N°"].apply(piso_cat)


# **Split de datos (train/test)**

### Separación de variables predictoras (X) y variable objetivo (y, "Precio")

### <span style="color: #dc2626;">!!! **Importante: comentarios para recordar**</span>


1. voy a usar Barrio_group como parte de las features, pero la idea es que luego podamos definir realmente cuales van a ser las variables que vamos a usar despues de que toda la limpieza y análisis haya terminado, estoy usando el que hace que vaya a "otros"
2. también estoy usando piso_cat
3. pareciera que para poder usar optuna, es mejor hacer algo como train, test y validation porque optuna corremos el riesgo de ajustar el modelo al test indirectamente, la idea es que  Optuna compare configuraciones sin “mirar” el test final, y eso es algo que no hicimos en el trabajo de la profe camila

70% train: el modelo aprende
15% validation : Optuna prueba distintas combinaciones y decide cuáles son mejores
15% test: una sola vez al final para medir el desempeño real

4. tenemos que definir que métrica nos importa mas para nuestros modelos entre MAE, RMSE y R2
5. tampoco podemos incluir area privada m2 porque hace multicolinealidad


In [5]:
df_filtrado.columns

Index(['ID', 'Barrio', 'Tipo de Inmueble', 'Estado', 'Antigüedad',
       'Área Construida (m2)', 'Área Privada (m2)', 'Estrato', 'Baños',
       'Habitaciones', 'Parqueaderos', 'Piso N°', 'URL', 'Precio',
       'Barrio_clean', 'Barrio_group', 'Piso_cat'],
      dtype='object')

In [6]:

numeric_features = [
    "Área Construida (m2)",
    "Estrato",
    "Baños",
    "Habitaciones",
    "Parqueaderos",
]

categorical_features = [
    "Tipo de Inmueble",
    "Estado",
    "Antigüedad",
    "Barrio_group",
    "Piso_cat",
]

features = numeric_features + categorical_features



In [7]:
# Separación de variables predictoras (X) y variable objetivo (y, "Precio")

X = df_filtrado[features]
y = df_filtrado["Precio"]

## 70% train, 30% temporal test

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
)  

# Del 30% temporal, mitad validation y mitad test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42
)

# Checamos el shape de las variables de entreno y prueba X, Y
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (3719, 10)
y_train: (3719,)
X_val: (797, 10)
y_val: (797,)
X_test: (798, 10)
y_test: (798,)


## **Pipeline: preprocesamiento + modelo (evitar data leakage)**

In [8]:
df_filtrado.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5314 entries, 0 to 5611
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ID                    5314 non-null   int64  
 1   Barrio                5314 non-null   object 
 2   Tipo de Inmueble      5314 non-null   object 
 3   Estado                5314 non-null   object 
 4   Antigüedad            5314 non-null   object 
 5   Área Construida (m2)  5314 non-null   float64
 6   Área Privada (m2)     5314 non-null   float64
 7   Estrato               5314 non-null   float64
 8   Baños                 5273 non-null   float64
 9   Habitaciones          5226 non-null   float64
 10  Parqueaderos          5314 non-null   int64  
 11  Piso N°               3157 non-null   float64
 12  URL                   5314 non-null   object 
 13  Precio                5314 non-null   int64  
 14  Barrio_clean          5314 non-null   object 
 15  Barrio_group          5314

### **Pipelines y transformaciones**

Modelos: 

1. LinearRegression
2. RandomForestRegressor
3. LGBMRegressor

- `PowerTransformer` para normalizar la distribución del Precio, usamos `Yeo-Johnson` porque a pesar de que nuestra variable `Precio` ya tiene valores positivos y podríamos usar `Box-Cox`, nos pareció mejor un metodo que fuera tan estricto, es mas flexible, y aunque ambos buscan redcir la asimetría y estabilizar la varianza, Box-Cox es estricamente mayores a 0, y queríamos una transformación mas segura y fácil de integrar en la pipeline

ademas, es mas flexible que hacer escala logarítimca `np.log1p` sobre nuestra y "precio" porque se adapta a los datos en vez de asumir siempre logartimo.

- PowerTransformer(y)  →  normaliza la distribución del PRECIO 
- RobustScaler(X)      →  escala numeric_features siendo robusto a outliers solo para LinearRegression porque en los de arboles el escalado no influye

Para el modelo lineal (`LinearRegression`) se usa **`RobustScaler`** en lugar de `StandardScaler`.

| Scaler | Fórmula | Problema |
|---|---|---|
| `StandardScaler` | `z = (x − media) / std` | La media y std se ven jaladas por outliers |
| `RobustScaler` | `z = (x − mediana) / IQR` | La mediana y el IQR son resistentes a outliers |

IQR (Rango Intercuartílico)** es la distancia entre el percentil 25 (Q1) y el percentil 75 (Q3).
Cubre el **50% central de los datos**, ignorando los extremos al calcular la escala.

In [9]:
areas = ["Área Construida (m2)", "Área Privada (m2)"]

df_filtrado[areas].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
Área Construida (m2),5314.0,106.319407,103.569580,1.0,20.0,35.0,60.0,80.0,115.0,280.0,480.0,2416.0
Área Privada (m2),5314.0,106.755237,106.914379,1.0,20.0,35.0,60.0,79.0,115.0,280.0,500.0,2416.0


<span style="color: orange;"><strong>Observación actualizada</strong></span>

`Área Privada` aún presenta valores máximos muy altos, por lo que el filtro debe aplicarse en ambas variables de área:

```python
df_filtrado = df_filtrado[
    (df_filtrado["Área Construida (m2)"] <= 2500) &
    (df_filtrado["Área Privada (m2)"] <= 2500)
].copy()
```


Aunque el filtro reduce bastante los valores extremos, la distribución de las áreas sigue siendo asimétrica.

Mediana de Área Construida (m2): 80 m²
Media: 106.7 m²
IQR: 115 - 60 = 55 m²

Esto sugiere que todavía hay valores altos que desplazan la media hacia arriba.
Por eso, StandardScaler seguiría centrando con respecto a una media afectada por extremos, mientras que RobustScaler usa la mediana y el rango intercuartílico (IQR), representando mejor el comportamiento típico de los inmuebles.

In [10]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Área Construida (m2)", "Área Privada (m2)")
)

fig.add_trace(
    go.Box(
        x=df_filtrado["Área Construida (m2)"],
        name="Área Construida",
        marker=dict(color="#0f766e"),
        fillcolor="rgba(15, 118, 110, 0.35)",
        line=dict(color="#0f766e"),
        boxmean=True,
        boxpoints="outliers",
        hovertemplate="Área construida: %{x:.2f} m²<extra></extra>"
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Box(
        x=df_filtrado["Área Privada (m2)"],
        name="Área Privada",
        marker=dict(color="#b45309"),
        fillcolor="rgba(180, 83, 9, 0.35)",
        line=dict(color="#b45309"),
        boxmean=True,
        boxpoints="outliers",
        hovertemplate="Área privada: %{x:.2f} m²<extra></extra>"
    ),
    row=1,
    col=2
)

fig.update_layout(
    title="Distribución y valores extremos en las variables de área",
    template="plotly_white",
    showlegend=False,
    width=1050,
    height=450,
    font=dict(size=13),
    margin=dict(t=70, l=40, r=40, b=40)
)

fig.update_xaxes(title_text="Metros cuadrados", row=1, col=1)
fig.update_xaxes(title_text="Metros cuadrados", row=1, col=2)

fig.show()


In [11]:
# Mira qué hay entre 500 y 4000
df[(df["Área Construida (m2)"] > 500) & 
   (df["Área Construida (m2)"] <= 4000)][["Barrio", "Tipo de Inmueble", "Área Construida (m2)", "Precio"]].sort_values("Área Construida (m2)", ascending=False).head(20)

,Barrio,Tipo de Inmueble,Área Construida (m2),Precio
1480,Buenos aires,Apartamento,4000.0,1150000
206,Castilla,Apartamento,4000.0,1000000
215,Castilla,Apartamento,4000.0,950000
217,Girardot,Apartaestudio,4000.0,1100000
254,Pedregal,Apartamento,4000.0,800000
703,Boston,Apartamento,3900.0,830000
705,Boston,Apartamento,3900.0,830000
704,Boston,Apartamento,3900.0,830000
1328,Buenos aires,Apartamento,3800.0,1200000
2166,Centro,Apartaestudio,3700.0,1100000


In [12]:
df_filtrado['Barrio_group'].value_counts()

Barrio_group
EL POBLADO          890
LAURELES            611
BELEN               288
CALASANZ            251
OTROS               205
                   ... 
BARRIO CRISTOBAL      4
LAS GRANJAS           4
MANRIQUE CENTRAL      4
GRANIZAL              2
ALFONSO LOPEZ         1
Name: count, Length: 115, dtype: int64

In [13]:
# Transformers

# ---------------------------Transformadores---------------------------

numeric_transformer_lr = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),  # imputamos faltantes numéricos con la mediana (que ya no deberíamos de tener)
        ("scaler", RobustScaler()),                   # escalado para modelos lineales
    ]
)

numeric_transformer_tree = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),  # los árboles no requieren escalado
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("encoder", TargetEncoder(smooth=10)),  # importante para barrios con pocas muestras, suaviza la media del target para evitar overfitting, toma el promedio global
    ]
)

# Usamos TargetEncoder porque resume las categorías según su relación con el target ("Precio"),
# y evita expandir demasiado la dimensionalidad cuando hay varias categorías.

# ---------------------------Preprocesadores---------------------------

# ---Modelo lineal---
lr_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_lr, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)

# ---Trees: RandomForest y LightGBM---
tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_tree, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)

# ---------------------------Pipelines---------------------------

lr_pipeline = Pipeline(
    steps=[
        ("preprocessor", lr_preprocessor),
        ("model", Ridge(alpha=1.0))
    ]
)

rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", tree_preprocessor),
        ("model", TransformedTargetRegressor(
            regressor=RandomForestRegressor(random_state=42),
            transformer=PowerTransformer(method="yeo-johnson")
        ))
    ]
)

lgbm_pipeline = Pipeline(
    steps=[
        ("preprocessor", tree_preprocessor),
        ("model", TransformedTargetRegressor(
            regressor=LGBMRegressor(random_state=42),
            transformer=PowerTransformer(method="yeo-johnson")
        ))
    ]
)


# **Entrenar: levantamos el MLflow Tracking Server**

```bash
mlflow server \
  --host 127.0.0.1 \
  --port 5001 \
  --backend-store-uri sqlite:///mlflow.db \
  --default-artifact-root ./mlruns
```
Opción que me funcionó con Powershell
```powershell
mlflow server `
  --host 127.0.0.1 `
  --port 5001 `
  --backend-store-uri sqlite:///mlflow.db `
  --default-artifact-root ./mlruns
```

`Puerto: http://127.0.0.1:5001`

Y se crea el file `mlflow.db`

In [14]:
import mlflow
mlflow.set_tracking_uri("http://127.0.0.1:5001")
print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

tracking URI: 'http://127.0.0.1:5001'


In [15]:
for col in categorical_features:
    counts = X_train[col].value_counts()
    raros = counts[counts == 1]
    if len(raros) > 0:
        print(f"{col}: {raros.index.tolist()}")

Barrio_group: ['GRANIZAL', 'ALFONSO LOPEZ']


# **LinearRegression**


In [16]:

mlflow.set_experiment("proyecto2-linear-regression")

with mlflow.start_run(run_name="baseline_linear_regression"):
    mlflow.set_tag("problem_type", "regression")
    mlflow.set_tag("model_family", "linear_regression")
    mlflow.set_tag("target", "Precio")
    mlflow.set_tag("features", ",".join(features))

    lr_pipeline.fit(X_train, y_train)

    y_pred_train = lr_pipeline.predict(X_train)
    y_pred_val = lr_pipeline.predict(X_val)
    y_pred_test = lr_pipeline.predict(X_test)

    print("NaN en y_pred_train:", np.isnan(y_pred_train).sum())
    print("NaN en y_pred_val:", np.isnan(y_pred_val).sum())
    print("NaN en y_pred_test:", np.isnan(y_pred_test).sum())
    print(y_pred_train[:10])



    train_mae = mean_absolute_error(y_train, y_pred_train)
    train_rmse = mean_squared_error(y_train, y_pred_train) ** 0.5
    train_r2 = r2_score(y_train, y_pred_train)

    val_mae = mean_absolute_error(y_val, y_pred_val)
    val_rmse = mean_squared_error(y_val, y_pred_val) ** 0.5
    val_r2 = r2_score(y_val, y_pred_val)

    test_mae = mean_absolute_error(y_test, y_pred_test)
    test_rmse = mean_squared_error(y_test, y_pred_test) ** 0.5
    test_r2 = r2_score(y_test, y_pred_test)

    mlflow.log_param("model", "LinearRegression")
    mlflow.log_param("numeric_features", numeric_features)
    mlflow.log_param("categorical_features", categorical_features)

    mlflow.log_metric("train_mae", train_mae)
    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("train_r2", train_r2)

    mlflow.log_metric("val_mae", val_mae)
    mlflow.log_metric("val_rmse", val_rmse)
    mlflow.log_metric("val_r2", val_r2)

    mlflow.log_metric("test_mae", test_mae)
    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("test_r2", test_r2)

    mlflow.sklearn.log_model(lr_pipeline, "model")

print("LinearRegression")
print("Train RMSE:", train_rmse)
print("Validation RMSE:", val_rmse)
print("Test RMSE:", test_rmse)
print("Test MAE:", test_mae)
print("Test R2:", test_r2)

print("=" * 40)
print("LinearRegression")
print("=" * 40)
print(f"Train      → RMSE: ${train_rmse:,.0f}  R²: {train_r2:.4f}")
print(f"Validation → RMSE: ${val_rmse:,.0f}  R²: {val_r2:.4f}")
print(f"Test       → RMSE: ${test_rmse:,.0f}  MAE: ${test_mae:,.0f}  R²: {test_r2:.4f}")


MlflowException: API request to http://127.0.0.1:5001/api/2.0/mlflow/experiments/get-by-name failed with exception HTTPConnectionPool(host='127.0.0.1', port=5001): Max retries exceeded with url: /api/2.0/mlflow/experiments/get-by-name?experiment_name=proyecto2-linear-regression (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=5001): Failed to establish a new connection: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión"))

# **RandomForestRegressor**

In [ ]:

mlflow.set_experiment("proyecto2-random-forest-optuna")
mlflow.autolog(log_models=False)

def objective_rf(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 15),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "random_state": 42,
        "n_jobs": -1,
    }

    rf_pipeline_optuna = Pipeline(
        steps=[
            ("preprocessor", tree_preprocessor),
            ("model", TransformedTargetRegressor(
                regressor=RandomForestRegressor(**params),
                transformer=PowerTransformer(method="yeo-johnson")
            ))
        ]
    )

    with mlflow.start_run(run_name=f"rf_trial_{trial.number}", nested=True) as run:
        trial.set_user_attr("mlflow_run_id", run.info.run_id)

        mlflow.set_tag("problem_type", "regression")
        mlflow.set_tag("model_family", "random_forest")
        mlflow.set_tag("target", "Precio")
        mlflow.set_tag("optimization", "optuna")
        mlflow.set_tag("features", ",".join(features))

        rf_pipeline_optuna.fit(X_train, y_train)
        y_pred_val = rf_pipeline_optuna.predict(X_val)

        mae = mean_absolute_error(y_val, y_pred_val)
        rmse = mean_squared_error(y_val, y_pred_val) ** 0.5
        r2 = r2_score(y_val, y_pred_val)

        mlflow.log_metric("val_mae", mae)
        mlflow.log_metric("val_rmse", rmse)
        mlflow.log_metric("val_r2", r2)

        return rmse


2026/05/10 08:21:02 INFO mlflow.tracking.fluent: Autologging successfully enabled for lightgbm.
2026/05/10 08:21:04 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


In [ ]:
study_rf = optuna.create_study(direction="minimize")

with mlflow.start_run(run_name="optuna_study_random_forest") as parent_run:
    mlflow.set_tag("stage", "hpo")

    study_rf.optimize(objective_rf, n_trials=10)

    top_trials = sorted(
        [t for t in study_rf.trials if t.value is not None],
        key=lambda t: t.value
    )[:5]

    top_trials_data = [
        {
            "trial_number": t.number,
            "rmse": t.value,
            "params": t.params,
            "mlflow_run_id": t.user_attrs.get("mlflow_run_id"),
        }
        for t in top_trials
    ]

    mlflow.log_params(study_rf.best_params)
    mlflow.log_metric("best_val_rmse", study_rf.best_value)
    mlflow.log_dict(top_trials_data, "optuna_top_trials_rf.json")
    mlflow.log_dict(study_rf.best_params, "optuna_best_params_rf.json")

print(f"Parent run id: {parent_run.info.run_id}")
print(f"Best params: {study_rf.best_params}")
print(f"Best validation RMSE: {study_rf.best_value}")


[I 2026-05-10 08:21:18,585] A new study created in memory with name: no-name-b0af44bd-60a6-42b1-9755-72aa8ace922b


2026/05/10 08:21:19 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWa

🏃 View run rf_trial_0 at: http://127.0.0.1:5001/#/experiments/2/runs/f01842c6abab4d91a5e544f7fd8c8f81
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2


2026/05/10 08:23:10 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWa

🏃 View run rf_trial_1 at: http://127.0.0.1:5001/#/experiments/2/runs/26f3087d38e342ec84f24f89f78ff440
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2


...

🏃 View run rf_trial_7 at: http://127.0.0.1:5001/#/experiments/2/runs/1448c02006234ba09457743a96955f95
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2


2026/05/10 08:26:45 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWa

🏃 View run rf_trial_8 at: http://127.0.0.1:5001/#/experiments/2/runs/9fd35d6ebc894203af1b37a0c7c879ce
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2


2026/05/10 08:28:30 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWa

🏃 View run rf_trial_9 at: http://127.0.0.1:5001/#/experiments/2/runs/9bfc3858b6f642299e08431de0fd1b3a
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2
🏃 View run optuna_study_random_forest at: http://127.0.0.1:5001/#/experiments/2/runs/c34a34ece5a04fb1a86b3151ad369d9c
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2
Parent run id: c34a34ece5a04fb1a86b3151ad369d9c
Best params: {'n_estimators': 401, 'max_depth': 9, 'min_samples_split': 7, 'min_samples_leaf': 2, 'max_features': None}
Best validation RMSE: 1839164.2971833746


In [ ]:
# Evaluación final del mejor RandomForest en test

best_rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", tree_preprocessor),
        ("model", TransformedTargetRegressor(
            regressor=RandomForestRegressor(**study_rf.best_params, random_state=42, n_jobs=-1),
            transformer=PowerTransformer(method="yeo-johnson")
        ))
    ]
)

best_rf_pipeline.fit(X_train, y_train)

y_pred_test_rf = best_rf_pipeline.predict(X_test)

test_mae_rf = mean_absolute_error(y_test, y_pred_test_rf)
test_rmse_rf = mean_squared_error(y_test, y_pred_test_rf) ** 0.5
test_r2_rf = r2_score(y_test, y_pred_test_rf)

print("RandomForestRegressor")
print("Test MAE:", test_mae_rf)
print("Test RMSE:", test_rmse_rf)
print("Test R2:", test_r2_rf)


2026/05/10 08:28:55 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '6df3d25999684646acce83bc5e673dd1', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/05/10 08:28:55 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling

🏃 View run trusting-eel-157 at: http://127.0.0.1:5001/#/experiments/2/runs/6df3d25999684646acce83bc5e673dd1
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2


2026/05/10 08:30:38 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


RandomForestRegressor
Test MAE: 879378.5772940622
Test RMSE: 1747435.0274049274
Test R2: 0.7825656594779201


In [ ]:
print(f"Best trial: {study_rf.best_trial.number}")
print(f"Best value (Validation RMSE): {study_rf.best_value:.4f}")
print("Best params:")
for key, value in study_rf.best_params.items():
    print(f"  {key}: {value}")

best_params_rf = {**study_rf.best_params, "random_state": 42, "n_jobs": -1}

best_pipeline_rf = Pipeline(
    steps=[
        ("preprocessor", tree_preprocessor),
        ("model", TransformedTargetRegressor(
            regressor=RandomForestRegressor(**best_params_rf),
            transformer=PowerTransformer(method="yeo-johnson")
        ))
    ]
)

best_pipeline_rf.fit(X_train, y_train)

y_pred_train_rf = best_pipeline_rf.predict(X_train)
y_pred_val_rf = best_pipeline_rf.predict(X_val)
y_pred_test_rf = best_pipeline_rf.predict(X_test)

train_mae_rf = mean_absolute_error(y_train, y_pred_train_rf)
train_rmse_rf = mean_squared_error(y_train, y_pred_train_rf) ** 0.5
train_r2_rf = r2_score(y_train, y_pred_train_rf)

val_mae_rf = mean_absolute_error(y_val, y_pred_val_rf)
val_rmse_rf = mean_squared_error(y_val, y_pred_val_rf) ** 0.5
val_r2_rf = r2_score(y_val, y_pred_val_rf)

test_mae_rf = mean_absolute_error(y_test, y_pred_test_rf)
test_rmse_rf = mean_squared_error(y_test, y_pred_test_rf) ** 0.5
test_r2_rf = r2_score(y_test, y_pred_test_rf)

print("\nRandomForestRegressor - mejor modelo")
print(f"Train MAE: {train_mae_rf:.4f}")
print(f"Train RMSE: {train_rmse_rf:.4f}")
print(f"Train R2: {train_r2_rf:.4f}")

print(f"\nValidation MAE: {val_mae_rf:.4f}")
print(f"Validation RMSE: {val_rmse_rf:.4f}")
print(f"Validation R2: {val_r2_rf:.4f}")

print("\n--- Métricas finales para comparar modelos ---")
print(f"Test MAE: {test_mae_rf:.4f}")
print(f"Test RMSE: {test_rmse_rf:.4f}")
print(f"Test R2: {test_r2_rf:.4f}")


Best trial: 8
Best value (Validation RMSE): 1839164.2972
Best params:
  n_estimators: 401
  max_depth: 9
  min_samples_split: 7
  min_samples_leaf: 2
  max_features: None


2026/05/10 08:32:04 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '9c4c258b976741e08f82a58ea2fd92f9', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/05/10 08:32:04 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling

🏃 View run big-elk-535 at: http://127.0.0.1:5001/#/experiments/2/runs/9c4c258b976741e08f82a58ea2fd92f9
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2


2026/05/10 08:33:50 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/10 08:33:50 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\elois\OneDr


RandomForestRegressor - mejor modelo
Train MAE: 783933.3664
Train RMSE: 1820260.8481
Train R2: 0.7983

Validation MAE: 934756.9678
Validation RMSE: 1898849.9965
Validation R2: 0.7642

--- Métricas finales para comparar modelos ---
Test MAE: 879470.1974
Test RMSE: 1738997.7355
Test R2: 0.7847


# **LGBMRegressor**

In [ ]:
mlflow.set_experiment("proyecto2-lightgbm-optuna")
mlflow.autolog(log_models=False)

def objective_lgbm(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 100),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "random_state": 42,
        "verbosity": -1,
    }

    lgbm_pipeline_optuna = Pipeline(
        steps=[
            ("preprocessor", tree_preprocessor),
            ("model", TransformedTargetRegressor(
                regressor=LGBMRegressor(**params),
                transformer=PowerTransformer(method="yeo-johnson")
            ))
        ]
    )

    with mlflow.start_run(run_name=f"lgbm_trial_{trial.number}", nested=True) as run:
        trial.set_user_attr("mlflow_run_id", run.info.run_id)

        mlflow.set_tag("problem_type", "regression")
        mlflow.set_tag("model_family", "lightgbm")
        mlflow.set_tag("target", "Precio")
        mlflow.set_tag("optimization", "optuna")
        mlflow.set_tag("features", ",".join(features))

        lgbm_pipeline_optuna.fit(X_train, y_train)
        y_pred_val = lgbm_pipeline_optuna.predict(X_val)

        mae = mean_absolute_error(y_val, y_pred_val)
        rmse = mean_squared_error(y_val, y_pred_val) ** 0.5
        r2 = r2_score(y_val, y_pred_val)

        mlflow.log_metric("val_mae", mae)
        mlflow.log_metric("val_rmse", rmse)
        mlflow.log_metric("val_r2", r2)

        return rmse


2026/05/10 08:34:50 INFO mlflow.tracking.fluent: Experiment with name 'proyecto2-lightgbm-optuna' does not exist. Creating a new experiment.
2026/05/10 08:34:50 INFO mlflow.tracking.fluent: Autologging successfully enabled for lightgbm.
2026/05/10 08:34:51 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


In [ ]:
study_lgbm = optuna.create_study(direction="minimize")

with mlflow.start_run(run_name="optuna_study_lightgbm") as parent_run:
    mlflow.set_tag("stage", "hpo")

    study_lgbm.optimize(objective_lgbm, n_trials=10)

    top_trials = sorted(
        [t for t in study_lgbm.trials if t.value is not None],
        key=lambda t: t.value
    )[:5]

    top_trials_data = [
        {
            "trial_number": t.number,
            "rmse": t.value,
            "params": t.params,
            "mlflow_run_id": t.user_attrs.get("mlflow_run_id"),
        }
        for t in top_trials
    ]

    mlflow.log_params(study_lgbm.best_params)
    mlflow.log_metric("best_val_rmse", study_lgbm.best_value)
    mlflow.log_dict(top_trials_data, "optuna_top_trials_lgbm.json")
    mlflow.log_dict(study_lgbm.best_params, "optuna_best_params_lgbm.json")

print(f"Parent run id: {parent_run.info.run_id}")
print(f"Best params: {study_lgbm.best_params}")
print(f"Best validation RMSE: {study_lgbm.best_value}")


[I 2026-05-10 08:35:05,178] A new study created in memory with name: no-name-e10a007d-4c39-4a49-a5b8-61277470250d
2026/05/10 08:35:05 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
c:\Users\

🏃 View run lgbm_trial_0 at: http://127.0.0.1:5001/#/experiments/3/runs/a7efd425df7d4f58bab7ec682dca4dfc
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/3


2026/05/10 08:35:16 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWa

🏃 View run lgbm_trial_1 at: http://127.0.0.1:5001/#/experiments/3/runs/9c477ccfe3ba4f2b8c1993f659a1a7a7
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/3


2026/05/10 08:35:33 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWa

...

🏃 View run lgbm_trial_7 at: http://127.0.0.1:5001/#/experiments/3/runs/9311f3199cab4292911ca629977e2eea
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/3


2026/05/10 08:37:27 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWa

🏃 View run lgbm_trial_8 at: http://127.0.0.1:5001/#/experiments/3/runs/c78fb2ef0134440c97d622b3ab3d8e34
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/3


2026/05/10 08:37:43 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWa

🏃 View run lgbm_trial_9 at: http://127.0.0.1:5001/#/experiments/3/runs/bc17e4073a504be0b12daf671bad4db0
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/3
🏃 View run optuna_study_lightgbm at: http://127.0.0.1:5001/#/experiments/3/runs/d020c62abf7440678e0eb614a18d847e
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/3
Parent run id: d020c62abf7440678e0eb614a18d847e
Best params: {'n_estimators': 183, 'learning_rate': 0.04980915502443387, 'num_leaves': 46, 'max_depth': 8, 'min_child_samples': 10, 'subsample': 0.7586900848965632, 'colsample_bytree': 0.730035407781606}
Best validation RMSE: 1743648.0755112562


In [ ]:

print(f"Best trial: {study_lgbm.best_trial.number}")
print(f"Best value (Validation RMSE): {study_lgbm.best_value:.4f}")
print("Best params:")
for key, value in study_lgbm.best_params.items():
    print(f"  {key}: {value}")

best_params_lgbm = {**study_lgbm.best_params, "random_state": 42, "verbosity": -1}

best_pipeline_lgbm = Pipeline(
    steps=[
        ("preprocessor", tree_preprocessor),
        ("model", TransformedTargetRegressor(
            regressor=LGBMRegressor(**best_params_lgbm),
            transformer=PowerTransformer(method="yeo-johnson")
        ))
    ]
)

best_pipeline_lgbm.fit(X_train, y_train)

y_pred_train_lgbm = best_pipeline_lgbm.predict(X_train)
y_pred_val_lgbm = best_pipeline_lgbm.predict(X_val)
y_pred_test_lgbm = best_pipeline_lgbm.predict(X_test)

train_mae_lgbm = mean_absolute_error(y_train, y_pred_train_lgbm)
train_rmse_lgbm = mean_squared_error(y_train, y_pred_train_lgbm) ** 0.5
train_r2_lgbm = r2_score(y_train, y_pred_train_lgbm)

val_mae_lgbm = mean_absolute_error(y_val, y_pred_val_lgbm)
val_rmse_lgbm = mean_squared_error(y_val, y_pred_val_lgbm) ** 0.5
val_r2_lgbm = r2_score(y_val, y_pred_val_lgbm)

test_mae_lgbm = mean_absolute_error(y_test, y_pred_test_lgbm)
test_rmse_lgbm = mean_squared_error(y_test, y_pred_test_lgbm) ** 0.5
test_r2_lgbm = r2_score(y_test, y_pred_test_lgbm)

print("\nLGBMRegressor - mejor modelo")
print(f"Train MAE: {train_mae_lgbm:.4f}")
print(f"Train RMSE: {train_rmse_lgbm:.4f}")
print(f"Train R2: {train_r2_lgbm:.4f}")

print(f"\nValidation MAE: {val_mae_lgbm:.4f}")
print(f"Validation RMSE: {val_rmse_lgbm:.4f}")
print(f"Validation R2: {val_r2_lgbm:.4f}")

print("\n--- Métricas finales para comparar modelos ---")
print(f"Test MAE: {test_mae_lgbm:.4f}")
print(f"Test RMSE: {test_rmse_lgbm:.4f}")
print(f"Test R2: {test_r2_lgbm:.4f}")


2026/05/10 08:38:54 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '6de657bf787a433fb4ae1de0f0d20558', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


Best trial: 2
Best value (Validation RMSE): 1743648.0755
Best params:
  n_estimators: 183
  learning_rate: 0.04980915502443387
  num_leaves: 46
  max_depth: 8
  min_child_samples: 10
  subsample: 0.7586900848965632
  colsample_bytree: 0.730035407781606


2026/05/10 08:38:54 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWa

🏃 View run wistful-bat-382 at: http://127.0.0.1:5001/#/experiments/3/runs/6de657bf787a433fb4ae1de0f0d20558
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/3


c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
2026/05/10 08:39:05 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <htt


LGBMRegressor - mejor modelo
Train MAE: 753341.1319
Train RMSE: 1698418.7853
Train R2: 0.8244

Validation MAE: 863960.4107
Validation RMSE: 1749334.6314
Validation R2: 0.7999

--- Métricas finales para comparar modelos ---
Test MAE: 837380.8469
Test RMSE: 1672314.2306
Test R2: 0.8009


In [ ]:
resultados_finales = pd.DataFrame({
    "Modelo": ["Ridge", "RandomForestRegressor", "LGBMRegressor"],
    "Test MAE": [test_mae, test_mae_rf, test_mae_lgbm],
    "Test RMSE": [test_rmse, test_rmse_rf, test_rmse_lgbm],
    "Test R2": [test_r2, test_r2_rf, test_r2_lgbm],
})

resultados_finales = resultados_finales.sort_values("Test RMSE").reset_index(drop=True)
resultados_finales

,Modelo,Test MAE,Test RMSE,Test R2
0,LGBMRegressor,8.373808e+05,1.672314e+06,0.800858
1,RandomForestRegressor,8.794702e+05,1.738998e+06,0.784660
2,Ridge,1.176914e+06,2.555697e+06,0.534902


In [ ]:
# Formatear para presentación
resultados_finales["Test MAE"] = resultados_finales["Test MAE"].apply(lambda x: f"${x:,.0f}")
resultados_finales["Test RMSE"] = resultados_finales["Test RMSE"].apply(lambda x: f"${x:,.0f}")
resultados_finales["Test R2"] = resultados_finales["Test R2"].apply(lambda x: f"{x:.4f}")

resultados_finales

,Modelo,Test MAE,Test RMSE,Test R2
0,LGBMRegressor,"$837,381","$1,672,314",0.8009
1,RandomForestRegressor,"$879,470","$1,738,998",0.7847
2,Ridge,"$1,176,914","$2,555,697",0.5349
